# FAISS

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader('speech.txt')
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size = 1000, chunk_overlap = 30)
docs = text_splitter.split_documents(documents)

In [2]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\nâ€¦'),
 Document(metadata={'source': 'speech.txt'}, page_content='â€¦\n\nIt will be all the easier for us to conduct

In [7]:
embeddings = OllamaEmbeddings(model = 'nomic-embed-text:latest')
db = FAISS.from_documents(docs, embeddings)
db 

In [10]:
### Querying
query = "How does the speaker describe the desired outcome of the war?"
docs = db.similarity_search(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

In [12]:
### Retriever

retriever = db.as_retriever()
docs = retriever.invoke(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

In [13]:
### Similarity score with search

docs_and_score = db.similarity_search_with_score(query)
docs_and_score

[(Document(id='531d423b-e3f2-4809-ab41-dacc8ae10122', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  np.float32(343.15894)),
 (Document(id='66f1984b-144e-47a2-857c-bc30b7e1c801', metadata={'source': 's

In [14]:
embedding_vectore = embeddings.embed_query(query)
embedding_vectore

[-0.30233967304229736,
 0.27922913432121277,
 -3.614079236984253,
 -0.9984495639801025,
 2.0198402404785156,
 1.4682198762893677,
 -1.0226274728775024,
 0.23474368453025818,
 0.4825284481048584,
 -0.18021398782730103,
 -0.06266578286886215,
 0.5966466069221497,
 0.8668273687362671,
 1.6237223148345947,
 2.0874414443969727,
 -0.7815162539482117,
 0.2585598826408386,
 -1.208501935005188,
 -0.6825523376464844,
 0.7263363003730774,
 -0.9006661176681519,
 -0.3936302363872528,
 0.09215741604566574,
 0.479488730430603,
 1.3442444801330566,
 0.4100087285041809,
 -0.5244418382644653,
 0.6706198453903198,
 -1.0714094638824463,
 -0.52565598487854,
 0.9149174690246582,
 0.062026794999837875,
 -0.05055413395166397,
 0.28155678510665894,
 -1.536308765411377,
 -1.9173249006271362,
 0.07941436767578125,
 1.334786057472229,
 0.3555867075920105,
 -1.4279004335403442,
 0.061724063009023666,
 -0.4963761568069458,
 0.12287071347236633,
 -1.5528450012207031,
 1.3664401769638062,
 -0.6891381740570068,
 -0.13

In [ ]:
docs_score = db.similarity_search_by_vector(embedding_vectore)
docs_score

[Document(id='531d423b-e3f2-4809-ab41-dacc8ae10122', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='66f1984b-144e-47a2-857c-bc30b7e1c801', metadata={'source': 'speech.txt'}, page_content='â

In [16]:
### Saving and loading
db.save_local("faiss_index")

In [18]:
new_db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
docs = new_db.similarity_search(query)
docs

[Document(id='531d423b-e3f2-4809-ab41-dacc8ae10122', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='66f1984b-144e-47a2-857c-bc30b7e1c801', metadata={'source': 'speech.txt'}, page_content='â

# ChromaDB

In [25]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
# from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader('speech.txt')
data = loader.load()
text_splitter = CharacterTextSplitter(chunk_size = 500, chunk_overlap = 0)
splits = text_splitter.split_documents(data)

Created a chunk of size 670, which is longer than the specified 500
Created a chunk of size 984, which is longer than the specified 500
Created a chunk of size 791, which is longer than the specified 500


In [29]:
embedding = OllamaEmbeddings(model = 'nomic-embed-text:latest')
vectordb = Chroma.from_documents(documents = splits, embedding = embedding)

In [31]:
query = "What does the speaker believe is the main reason the United States should enter the war?"
docs = vectordb.similarity_search(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

In [35]:
## Saving locally
vectordb = Chroma.from_documents(documents=splits, embedding = embedding, persist_directory = './chroma_db')

In [36]:
## Load db
new_db = Chroma(persist_directory="./chroma_db", embedding_function=embedding)
docs = new_db.similarity_search(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

In [37]:
new_db.similarity_search_with_score(query)

[(Document(id='f7a80d51-79e1-429a-9145-b168654e0295', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  315.2855529785156),
 (Document(id='287376e6-3e7e-41a0-b433-f32fa57415af', metadata={'source': 'speec

In [38]:
## Retirever

retriever = new_db.as_retriever()
retriever.invoke(query)[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'